In [1]:
import pandas as pd
import numpy as np
import pickle
from difflib import get_close_matches
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer

In [2]:
def cosine_similarity(vector, matrix):
    norm_v = np.sqrt(np.dot(vector, vector))
    norm_m = np.sqrt(np.sum(matrix * matrix, axis=1))
    dot = np.dot(matrix, vector)
    return dot / (norm_m * norm_v + 1e-8)

In [3]:
df = pd.read_csv('./TMDB_movie_dataset_v11.csv')

In [4]:
to_drop = [
    'id', 'status', 'original_title', 'homepage', 'poster_path',
    'tagline', 'vote_count', 'release_date', 'revenue', 'backdrop_path',
    'budget', 'imdb_id', 'keywords', 'spoken_languages', 'overview',
    'production_companies', 'production_countries'
]
df = df.drop(to_drop, axis='columns')

In [5]:
df.dropna(subset=['title'], inplace=True)
df.fillna('Unknown', inplace=True)

,title,vote_average,runtime,adult,original_language,popularity,genres
0,Inception,8.364,148,False,en,83.9520,"Action, Science Fiction, Adventure"
1,Interstellar,8.417,169,False,en,140.2410,"Adventure, Drama, Science Fiction"
2,The Dark Knight,8.512,152,False,en,130.6430,"Drama, Action, Crime, Thriller"
3,Avatar,7.573,162,False,en,79.9320,"Action, Adventure, Fantasy, Science Fiction"
4,The Avengers,7.710,143,False,en,98.0820,"Science Fiction, Action, Adventure"
...,...,...,...,...,...,...,...
1390396,"Tonino De Bernardi: One Time, One Encounter",0.000,52,False,it,0.6000,Documentary
1390397,Shadow,0.000,35,False,en,0.8910,Unknown
1390398,Trastwest,0.000,18,False,en,0.6000,Unknown
1390399,American Style,0.000,89,False,en,0.6000,Unknown


In [6]:
df.adult = df.adult.apply(lambda x: 1 if x else 0)

In [7]:
df.drop_duplicates(subset=['title'], inplace=True)
df.drop_duplicates(inplace=True)

In [8]:
scaler = StandardScaler()
to_scale = ['vote_average', 'runtime', 'popularity']
df[to_scale] = scaler.fit_transform(df[to_scale])

In [9]:
df.genres = df.genres.apply(
    lambda x: [g.strip().lower().replace(' ', '') for g in x.split(',')]
)
df.title = df.title.apply(lambda x: x.strip().title())

In [10]:
mlb = MultiLabelBinarizer()
binary_genre = pd.DataFrame(
    mlb.fit_transform(df.genres),
    columns=mlb.classes_,
    index=df.index
)

In [11]:
lang_dummies = pd.get_dummies(df['original_language'], dtype=int)

In [12]:
feature_df = pd.concat([
    df[['title', 'vote_average', 'runtime', 'adult', 'popularity']],
    binary_genre,
    lang_dummies
], axis='columns')

In [13]:
feature_df = feature_df.iloc[:50000].reset_index(drop=True)

In [14]:
titles = feature_df['title']
feature_matrix = feature_df.drop(['title'], axis='columns').values.astype(np.float64)
print(feature_matrix.shape)

(50000, 200)


In [15]:
idx = titles[titles == 'Inception'].index[0]
vec = feature_matrix[idx]
sims = cosine_similarity(vec, feature_matrix)
top = np.argsort(sims)[::-1][1:11]
titles.iloc[top].tolist()

['Iron Man',
 'Mad Max: Fury Road',
 'Avengers: Endgame',
 'Captain America: Civil War',
 'Star Wars',
 'Star Wars: The Force Awakens',
 'Guardians Of The Galaxy Vol. 2',
 'Iron Man 2',
 'The Avengers',
 'Avengers: Age Of Ultron']

In [16]:
def suggest_movie(name, sugg_number=3):
    name = name.strip().title()
    if name not in titles.values:
        matches = get_close_matches(name, titles, n=1)
        if not matches:
            return f"Movie '{name}' not found in dataset"
        name = matches[0]
    idx = titles[titles == name].index[0]
    vec = feature_matrix[idx]
    sims = cosine_similarity(vec, feature_matrix)
    top = np.argsort(sims)[::-1][1:sugg_number+1]
    return titles.iloc[top].tolist()

In [17]:
print(suggest_movie('GodFather', 10))

['Punjabi House', 'Kumbalangi Nights', 'Udayananu Tharam', 'Thoovanathumbikal', 'Koode', 'Ramji Rao Speaking', 'Punyalan Agarbattis', 'Annayum Rasoolum', 'Mayaanadhi', "Salt N' Pepper"]


In [18]:
data = {'titles': titles, 'feature_matrix': feature_matrix}
with open('similarity.pkl', 'wb') as f:
    pickle.dump(data, f)